In [2]:
import pandas as pd

train = pd.read_csv('/home/jack/github/kaggle/scoring/xlnet_hash.csv')   

In [3]:
train.head()

,essay_id,clean_text,score,deberta_prob_0,deberta_prob_1,deberta_prob_2,deberta_prob_3,deberta_prob_4,deberta_prob_5,labels,...,hash_190,hash_191,hash_192,hash_193,hash_194,hash_195,hash_196,hash_197,hash_198,hash_199
0,000fe60,i am a scientist at nasa that is discussing th...,3,0.015906,0.832160,0.148738,0.002510,0.000311,0.000375,2,...,0.000000,0.119048,0.000000,0.000000,0.000000,0.023810,0.190476,0.071429,0.000000,0.02381
1,001ab80,people always wish they had the same technolog...,4,0.001099,0.010704,0.344778,0.631197,0.011407,0.000815,3,...,0.029130,0.043694,0.014565,0.014565,0.000000,0.014565,0.262167,0.014565,0.000000,0.00000
2,002ba53,dear state senator this is a letter to argue i...,3,0.059430,0.691130,0.233244,0.014120,0.001189,0.000887,2,...,0.000000,0.031758,0.000000,0.000000,0.000000,0.000000,0.381096,0.000000,0.031758,0.00000
3,0033037,the posibilty of a face reconizing computer wo...,2,0.110127,0.875540,0.012530,0.000799,0.000427,0.000577,1,...,0.000000,0.039073,0.000000,0.000000,0.000000,0.000000,0.195366,0.000000,0.000000,0.00000
4,0033bf4,what is the seagoing cowboys progam it was to ...,3,0.004650,0.469768,0.515730,0.009045,0.000419,0.000389,2,...,0.019364,0.019364,0.000000,0.019364,0.019364,0.000000,0.096819,0.193637,0.000000,0.00000


In [5]:
feature_cols = []

for col in train.columns:
    if (col != 'essay_id') and (col != 'score') and (col != 'labels') and (col != 'clean_text'):
        feature_cols.append(col) 

In [6]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
import numpy as np

# Assuming 'feature_cols' are defined elsewhere in your script

# Labels (stays the same)
train_labels = np.array(train['labels'])



# Features
train_features = train[feature_cols]




# Transform all datasets using the fitted scaler
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(train_features, train_labels, test_size=0.2, random_state=42)

# Initialize scaler
scaler = MinMaxScaler()

# Fit scaler to the original full training data
scaler.fit(X_train)


X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)

# # Save the scaler for later use
# with open('_scaler.pkl', 'wb') as f:
#     pickle.dump(scaler, f)


In [7]:
from sklearn.metrics import make_scorer, cohen_kappa_score

def quadratic_weighted_kappa_scorer(y_true, y_pred):
    """
    Compute the Quadratic Weighted Kappa (QWK), also known as Cohen's kappa.
    
    Parameters:
    y_true : array-like of shape (n_samples,)
        True labels.
    y_pred : array-liimport keras_tuner
from sklearn import ensemble
from sklearn import datasets
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection

def build_model(hp):
  model_type = hp.Choice('model_type', ['random_forest', 'ridge'])
  if model_type == 'random_forest':
    model = ensemble.RandomForestClassifier(
        n_estimators=hp.Int('n_estimators', 10, 50, step=10),
        max_depth=hp.Int('max_depth', 3, 10))
  else:
    model = linear_model.RidgeClassifier(
        alpha=hp.Float('alpha', 1e-3, 1, sampling='log'))
  return model

tuner = keras_tuner.tuners.SklearnTuner(
    oracle=keras_tuner.oracles.BayesianOptimizationOracle(
        objective=keras_tuner.Objective('score', 'max'),
        max_trials=100),
    hypermodel=build_model,
    scoring=metrics.make_scorer(metrics.accuracy_score),
    cv=model_selection.StratifiedKFold(10),
    directory='.',
    project_name='my_project')ke of shape (n_samples,)
        Predicted labels.
    
    Returns:
    score : float
        Quadratic Weighted Kappa score.
    """
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

In [8]:
qwk_scorer = make_scorer(quadratic_weighted_kappa_scorer)

In [11]:
import lazypredict
from lazypredict.Supervised import LazyClassifier
import pandas as pd


# Create and fit the LazyClassifier

clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric= None)    #quadratic_weighted_kappa_scorer)

models, predictions = clf.fit(X_train, X_val, y_train, y_val)

# Display the performance metrics of the models
print(models)


 97%|█████████▋| 28/29 [02:53<00:05,  5.33s/it]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003829 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 54060
[LightGBM] [Info] Number of data points in the train set: 13845, number of used features: 212
[LightGBM] [Info] Start training from score -2.610084
[LightGBM] [Info] Start training from score -1.305102
[LightGBM] [Info] Start training from score -1.014893
[LightGBM] [Info] Start training from score -1.478619
[LightGBM] [Info] Start training from score -2.882816
[LightGBM] [Info] Start training from score -4.715398
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


100%|██████████| 29/29 [02:55<00:00,  6.05s/it]

                               Accuracy  Balanced Accuracy ROC AUC  F1 Score  \
Model                                                                          
LinearDiscriminantAnalysis         0.70               0.72    None      0.70   
NearestCentroid                    0.59               0.68    None      0.59   
LogisticRegression                 0.71               0.67    None      0.71   
LinearSVC                          0.70               0.67    None      0.70   
XGBClassifier                      0.72               0.66    None      0.72   
LGBMClassifier                     0.73               0.65    None      0.72   
GaussianNB                         0.56               0.64    None      0.55   
BaggingClassifier                  0.70               0.64    None      0.70   
SVC                                0.70               0.64    None      0.70   
BernoulliNB                        0.63               0.63    None      0.63   
RidgeClassifier                    0.66 

In [1]:
import sklearn
print(sklearn.__version__)


1.4.2


In [10]:
# pandas display all the colmns in pd.columns
pd.set_option('display.max_columns', None)
